# Cas technique — IA appliquée à la radiographie thoracique

**Poste :** Data Scientist / ML Engineer — **Durée indicative :** 2 à 3 jours de travail effectif (à rendre sous 10 jours)

Une startup veut utiliser l'IA pour **détecter des anomalies sur des radiographies du thorax** et **pré-remplir un compte rendu (CR)** que le radiologue relit. Vous disposez de radiographies et des comptes rendus en texte libre rédigés par des radiologues. **Il n'y a pas de labels : la vérité terrain doit être extraite des CR.**

> **Ce qui est évalué :** la qualité et la justification de votre démarche, bien plus que le score final. Un modèle modeste mais rigoureusement évalué vaut mieux qu'un score élevé obtenu avec une fuite de données.

### Conventions
- 📦 **Cellule fournie** · 🧑‍💻 **À compléter** (structure libre) · ❓ **Question** (réponse en Markdown, concise et argumentée).
- Toute librairie et tout modèle pré-entraîné sont autorisés : citez vos sources et vérifiez l'absence de fuite. Assistants IA autorisés, mais vous devez pouvoir expliquer chaque choix au débrief.

## Partie 0 — Contexte et cadrage

Le client imagine une architecture **hiérarchique à deux niveaux** :

| Niveau | Tâche | Exigence |
|---|---|---|
| **N1** | Normal vs anormal (binaire) | Doit être **plus fiable** que N2. C'est le filtre de triage : un faux négatif renvoie un patient malade avec un CR « normal ». Cible annoncée « ≈ 97 % » (métrique non précisée). |
| **N2** | Présence de 13 pathologies (+ « No Finding ») | **Multi-label probabiliste** et **calibré** (ex. « 95 % pneumonie, 90 % nodule »). Les dépendances entre pathologies doivent être prises en compte. |

```mermaid
flowchart TD
    RADIO["Radiographie<br/>DICOM ~2K px → JPG"]
    VUES["Incidences multiples<br/>Face PA/AP + Profil<br/><i>(parfois absentes)</i>"]
    CLIN["Métadonnées + données cliniques<br/><i>(partielles)</i>"]
    PREP["Prétraitement<br/>Redimensionnement 512 → ~724 px"]
    CNN["Encodeur image<br/>CNN / Transformer"]
    RADIO --> PREP
    VUES --> PREP
    PREP --> CNN
    N1{"Niveau 1<br/>Anomalie ?<br/>cible ≈ 97 %"}
    CNN --> N1
    N1 -- NON --> CRN["CR : « normal »"]
    N1 -- OUI --> N2["Niveau 2"]
    N2 --> PN["PN : pathologies nommées<br/>13 pathologies<br/>classement ①, ②…"]
    N2 --> PNN["PNN : cas incertain<br/>« peut-être » pneumo, nodule…"]
    PNN --> CRR["CR revu par le médecin"]
    PN --> BN["Fusion bayésienne<br/>dépendances entre pathologies<br/>ex. nodule ↔ pneumonie"]
    CLIN --> BN
    BN --> OUT["Sortie multi-label probabiliste<br/>95 % pneumonie<br/>90 % nodule<br/>…"]
    OUT --> CRG["Ébauche de CR<br/>(aide à la relecture)"]
    classDef ok fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20
    classDef warn fill:#fff3e0,stroke:#ef6c00,color:#e65100
    classDef key fill:#e3f2fd,stroke:#1565c0,color:#0d47a1
    class CRN ok
    class PNN,CRR warn
    class N1,BN key
```

Contraintes : un examen peut avoir **0, 1 ou plusieurs vues** ; les données cliniques sont **partielles** ; le système doit **accepter nativement l'absence d'information**, sans imputation naïve.

### Données fournies (`donnees/`)

| Fichier | Contenu |
|---|---|
| `examens.csv` | 1 ligne par examen : `subject_id` (patient), `study_id` (examen), date/heure, nombre d'images, fichier du CR, présence d'un label radiologue |
| `images.csv` | 1 ligne par image : identifiants, **métadonnées DICOM** (`ViewPosition` AP / PA / LATERAL / LL, orientation du patient, description de l'acquisition, taille) — **partiellement manquantes** |
| `images/` | radiographies JPG (redimensionnées depuis le DICOM) |
| `comptes_rendus/` | 1 CR en texte libre (anglais) par examen : `INDICATION` / `HISTORY`, `COMPARISON`, `FINDINGS`, `IMPRESSION`… Les `___` remplacent des informations anonymisées |
| `labels_automatiques_chexpert.csv`, `labels_automatiques_negbio.csv` | labels extraits automatiquement des CR par deux outils publics (CheXpert labeler, NegBio) : 1 / 0 / -1 (incertain) / vide |
| `labels_radiologue.csv` | **labels posés par un radiologue** sur un sous-ensemble d'examens (vérité terrain de meilleure qualité) |

> ⚠️ Données de santé pseudonymisées, transmises **uniquement pour ce test** : ne les redistribuez pas, ne les poussez pas sur GitHub, supprimez-les après le débrief.

❓ **Q0.1** — Le client annonce une cible « ≈ 97 % » pour N1. Quelle(s) métrique(s) proposez-vous de lui présenter à la place (ou en plus), et pourquoi ?

❓ **Q0.2** — Listez les principaux risques de ce projet : cliniques, biais, fuite de données. Lesquels allez-vous pouvoir contrôler dans ce notebook ?

*Vos réponses :*

**Q0.1** —

**Q0.2** —

### Configuration
📦 Cellule fournie. `MODE_RAPIDE` réduit la résolution pour tourner sur CPU ; indiquez dans votre rendu le mode utilisé.

In [ ]:
import os, re, glob, json, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
warnings.filterwarnings("ignore")
import logging; logging.getLogger("pgmpy").setLevel(logging.ERROR)

DATA_DIR    = Path(os.environ.get("DATA_DIR", "donnees"))
REPORTS_DIR = DATA_DIR / "comptes_rendus"       # comptes rendus .txt
IMAGES_DIR  = DATA_DIR / "images"               # radiographies .jpg
CACHE_DIR   = Path(os.environ.get("CACHE_DIR", "cache")); CACHE_DIR.mkdir(exist_ok=True)

MODE_RAPIDE = os.environ.get("MODE_RAPIDE", "1") == "1"
SAMPLE_SIZE = None                              # nombre d'examens à garder (None = tous ; le jeu est petit)
IMG_SIZE    = 320 if MODE_RAPIDE else 512       # côté de l'image en entrée de l'encodeur
SEED = 42

def fixer_graines(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
fixer_graines()
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Mode rapide={MODE_RAPIDE} | examens={SAMPLE_SIZE or 'tous'} | IMG_SIZE={IMG_SIZE} | device={DEVICE}")
assert REPORTS_DIR.exists() and IMAGES_DIR.exists(), "Données introuvables : placez le dossier `donnees/` à la racine (voir README.md)"

### Chargement
📦 Cellule fournie : lecture brute des fichiers. Le reste (comptes rendus, jointures, vues…) est à votre charge.

In [ ]:
examens = pd.read_csv(DATA_DIR / "examens.csv")
images  = pd.read_csv(DATA_DIR / "images.csv")
lab_chexpert   = pd.read_csv(DATA_DIR / "labels_automatiques_chexpert.csv").set_index("study_id")
lab_negbio     = pd.read_csv(DATA_DIR / "labels_automatiques_negbio.csv").set_index("study_id")
lab_radiologue = pd.read_csv(DATA_DIR / "labels_radiologue.csv").set_index("study_id")
print(f"{len(examens)} examens, {len(images)} images, {len(lab_radiologue)} examens labellisés par un radiologue")

## Partie 1 — Exploration des données (EDA)

🧑‍💻 Explorez les images, les comptes rendus, les métadonnées et les labels fournis. **Signalez tout ce qui doit influencer la suite.**

In [ ]:
# 🧑‍💻 À compléter

❓ **Q1** — Quels constats de l'EDA vont directement contraindre vos choix dans la suite (labels, split, prétraitement, fusion) ?

*Votre réponse :*

## Partie 2 — Construction de la vérité terrain à partir des CR (NLP)

Référentiel de labels : **CheXpert** (13 pathologies + `No Finding`) :

`Enlarged Cardiomediastinum, Cardiomegaly, Lung Opacity, Lung Lesion, Edema, Consolidation, Pneumonia, Atelectasis, Pneumothorax, Pleural Effusion, Pleural Other, Fracture, Support Devices`

🧑‍💻 À faire :
1. Un **extracteur de labels** : positif / négatif / incertain / non mentionné, par examen et par label.
2. Sa **validation**, avec les ressources fournies.
3. Le **label N1 « anormal »**, cohérent avec N2.
4. Une **stratégie pour les labels incertains**, justifiée.

In [ ]:
PATHOLOGIES = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
               "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
               "Pleural Other", "Fracture", "Support Devices"]
LABELS = ["No Finding"] + PATHOLOGIES

In [ ]:
# 🧑‍💻 À compléter

❓ **Q2.1** — Quelle stratégie avez-vous retenue pour les labels incertains, et quel effet attendez-vous sur l'entraînement et sur l'évaluation ?

❓ **Q2.2** — Que vous apprend la comparaison avec les labels radiologue et avec CheXpert / NegBio sur la fiabilité de votre vérité terrain ? Quelles pathologies vous semblent trop bruitées pour être évaluées sérieusement ?

*Vos réponses :*

**Q2.1** —

**Q2.2** —

## Partie 3 — Nettoyage et préparation

🧑‍💻 À faire :
1. **Prétraitement des images** (résolution : le client parle de 512 → ~724 px).
2. **Identification de la vue** (face / profil).
3. **Split train / validation / test**.
4. **Déséquilibre des classes**.

Choix de l'encodeur image libre (pré-entraîné, gelé ou fine-tuné) : justifiez-le au regard du volume de données et de votre machine.

In [ ]:
# 🧑‍💻 À compléter

❓ **Q3.1** — Votre split garantit-il l'absence de fuite ? Que reste-t-il comme risque, et comment l'avez-vous réduit ?

❓ **Q3.2** — Quelle résolution d'entrée recommanderiez-vous au client (512, 724, 768…) et pourquoi ?

*Vos réponses :*

**Q3.1** —

**Q3.2** —

## Partie 4 — Modèle niveau 1 (normal / anormal)

🧑‍💻 À faire : un modèle N1, un **seuil** justifié par le coût des erreurs, une évaluation adaptée au contexte médical. Comparez ensuite avec un N1 **dérivé de N2** (Partie 5).

In [ ]:
# 🧑‍💻 À compléter

❓ **Q4** — Comment avez-vous choisi votre sensibilité cible et votre seuil ? Que répondez-vous au client qui demande « 97 % » ?

*Votre réponse :*

## Partie 5 — Modèle niveau 2 (multi-label, 13 pathologies)

🧑‍💻 À faire : un modèle N2 multi-label, son évaluation par pathologie, et des **probabilités calibrées** (vérifiées).

In [ ]:
# 🧑‍💻 À compléter

❓ **Q5** — Vos probabilités N2 sont-elles exploitables telles quelles par un médecin (« 95 % pneumonie ») ? Quelles pathologies ne devriez-vous **pas** afficher au client en l'état ?

*Votre réponse :*

## Partie 6 — Multi-vues et données manquantes

🧑‍💻 À faire : une fusion qui gère un **nombre variable de vues** et des **données cliniques / métadonnées partielles**, sans imputation naïve.

**Expérience obligatoire** : retirez artificiellement une vue, puis les données cliniques, et mesurez la dégradation.

In [ ]:
# 🧑‍💻 À compléter

❓ **Q6** — Pourquoi votre méthode de fusion « accepte-t-elle nativement » l'absence d'une vue ou d'une donnée clinique ? En quoi est-ce différent d'une imputation ?

*Votre réponse :*

## Partie 7 — Fusion bayésienne (CNN + réseau bayésien + clinique)

Les pathologies ne sont pas indépendantes. Le client imagine un **réseau bayésien** en couche de fusion (voir schéma, Partie 0), qui combine les sorties du modèle image, les dépendances entre pathologies et les données cliniques, avec des informations parfois absentes.

🧑‍💻 À faire : **mesurez** les dépendances, **construisez** cette fusion, et **comparez-la** à la baseline sans réseau bayésien. Vous pouvez défendre une alternative si vous l'argumentez. Librairie suggérée : `pgmpy`.

In [ ]:
# 🧑‍💻 À compléter

❓ **Q7.1** — Quelle structure avez-vous retenue et comment la défendriez-vous devant un radiologue ?

❓ **Q7.2** — Le réseau bayésien apporte-t-il réellement quelque chose par rapport à la fusion neuronale ? Sur quels critères (AUC, calibration, robustesse, interprétabilité) ?

*Vos réponses :*

**Q7.1** —

**Q7.2** —

## Partie 8 — Restitution et interprétabilité

🧑‍💻 À faire : une **sortie par examen** au format du client (P(anormal), pathologies probables, cas « à relire »), une **ébauche de CR** présentée comme aide à la relecture, et des **cartes d'activation** sur quelques cas, réussites et échecs.

In [ ]:
# 🧑‍💻 À compléter

❓ **Q8** — Que montrent vos cartes d'activation ? Feriez-vous confiance à ce modèle sur un cas où la carte est hors du poumon ?

*Votre réponse :*

## Partie 9 — Analyse critique

❓ **Q9.1** — Quelles sont les principales limites de votre travail (labels, données, protocole, modèle) ? Classez-les par impact.

❓ **Q9.2** — Que faudrait-il **avant un usage clinique** de ce système ? (validation, réglementaire, organisation, suivi en production)

❓ **Q9.3** — Si vous aviez deux semaines de plus et un GPU, que feriez-vous en priorité, et pourquoi ?

*Vos réponses :*

**Q9.1** —

**Q9.2** —

**Q9.3** —

---
### Rendu
Avant de rendre : `Kernel → Restart & Run All` doit tourner sans erreur dans le mode indiqué. Poussez le notebook **avec ses sorties**, puis suivez les instructions de rendu du `README.md`.